# Extract a bow-shock surface

This notebook reads a BATSRUS Tecplot file, identifies the bow shock from velocity compression, refines and smooths the extracted X surface, then plots the surface and normal angle on the Y-Z plane.

From the repository root:

```bash
pip install -e ".[notebook]"
jupyter lab examples/extract_shock.ipynb
```

The local `data/3d.dat` sample is about 1.3 GB and is not committed.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from shocklink.bowshock import (
    calc_bow_shock_normal_angle,
    calc_bow_shock_normals,
    extract_shockfit_range,
    fit_bow_shock,
    get_bow_shock_surface,
    smooth_bow_shock_surface,
)
from shocklink.dataset import calc_velocity_divergence
from shocklink.tecplot import read_tecplot


In [ ]:
DATA_PATH = Path("data/3d.dat")
SURFACE_Y = np.linspace(-20.0, 20.0, 81)
SURFACE_Z = np.linspace(-20.0, 20.0, 81)
SURFACE_X_RANGE = (-40.0, 20.0)
SMOOTHING_SIGMA = 2.0
REFERENCE_VECTOR = np.array([-1.0, 0.0, 0.0])

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Tecplot sample not found: {DATA_PATH}")


In [ ]:
grid = read_tecplot(DATA_PATH)
calc_velocity_divergence(grid)
fit = fit_bow_shock(grid)

shock_region = extract_shockfit_range(
    grid,
    lower=3.0 - fit.loc0[0],
    upper=fit.loc0[0] + 5.0,
)

print(f"Nose: {fit.loc0[0]:.3f} R; curvature: {fit.curvature:.4f}")
print(f"Shock-region cells: {shock_region.n_cells:,}")


In [ ]:
surface_x_raw = get_bow_shock_surface(
    shock_region,
    y=SURFACE_Y,
    z=SURFACE_Z,
    x_range=SURFACE_X_RANGE,
    refine_minimum=True,
)
surface_x = smooth_bow_shock_surface(
    surface_x_raw,
    sigma=SMOOTHING_SIGMA,
)
normals = calc_bow_shock_normals(surface_x, y=SURFACE_Y, z=SURFACE_Z)
normal_angle_deg = calc_bow_shock_normal_angle(normals, REFERENCE_VECTOR)

assert surface_x.shape == (len(SURFACE_Y), len(SURFACE_Z))
assert normal_angle_deg.shape == surface_x.shape
print(f"Finite surface points: {np.isfinite(surface_x).sum():,}/{surface_x.size:,}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")

surface_plot = axes[0].pcolormesh(
    SURFACE_Y, SURFACE_Z, surface_x.T, shading="auto", cmap="viridis"
)
axes[0].set(title="Smoothed bow-shock X surface", xlabel="Y [R]", ylabel="Z [R]")
axes[0].set_aspect("equal")
fig.colorbar(surface_plot, ax=axes[0], label="Bow-shock X [R]")

angle_plot = axes[1].pcolormesh(
    SURFACE_Y, SURFACE_Z, normal_angle_deg.T, shading="auto", cmap="magma"
)
axes[1].set(title="Angle to reference vector", xlabel="Y [R]", ylabel="Z [R]")
axes[1].set_aspect("equal")
fig.colorbar(angle_plot, ax=axes[1], label="Angle to reference vector [deg]")
plt.show()
